# Single-task learning: multilingual monolabel NER

This notebook trains one **XLM-RoBERTa** token classification model on multilingual disease NER data (IOB), merged into a single task (single classification head).

**Reference:** [Token classification (Hugging Face)](https://huggingface.co/docs/transformers/tasks/token_classification)


## Dependencies


In [ ]:
%pip install -q transformers datasets evaluate seqeval accelerate


## Imports


In [ ]:
import pandas as pd
import torch
import wandb
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments)


## Hugging Face Hub login

Authenticate if you want to push the trained model to the Hub.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## Data downloading

Download MultiClinAI Shared Task Training Set for MultiClinNER from [Zenodo](https://zenodo.org/records/19098018).


## Data uploading and formatting

Place the CSVs on Drive (or update paths). Expected layout (**IOB**), one row per token; sentence boundaries use blank lines / NaN tokens:

- **Disease** corpus: `/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/{cz,en,es,nl,sv,ro,it}/disease/kfold_5/{fold_*}/{train,val}/{lang}-disease-{fold}-{split}.csv`

- **Symptom** corpus: `/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/{cz,en,es,nl,sv,ro,it}/symptom/kfold_5/{fold_*}/{train,val}/{lang}-symptom-{fold}-{split}.csv`

- **Procedure** corpus: `/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/{cz,en,es,nl,sv,ro,it}/procedure/kfold_5/{fold_*}/{train,val}/{lang}-procedure-{fold}-{split}.csv`

Then, we connect to google drive, where we have stored our data:

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## CSV paths


In [ ]:
# Training data
train_en_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/disease/kfold_5/fold_1/train/en-disease-fold_1-train.csv"
train_nl_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/disease/kfold_5/fold_1/train/nl-disease-fold_1-train.csv"
train_sv_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/disease/kfold_5/fold_4/train/sv-disease-fold_4-train.csv"
train_es_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/disease/kfold_5/fold_1/train/es-disease-fold_1-train.csv"
train_it_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/disease/kfold_5/fold_1/train/it-disease-fold_1-train.csv"
train_ro_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/disease/kfold_5/fold_1/train/ro-disease-fold_1-train.csv"
train_cz_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/disease/kfold_5/fold_1/train/cz-disease-fold_1-train.csv"

# Validation / dev splits
dev_en_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/disease/kfold_5/fold_1/val/en-disease-fold_1-val.csv"
dev_nl_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/disease/kfold_5/fold_1/val/nl-disease-fold_1-val.csv"
dev_sv_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/disease/kfold_5/fold_4/val/sv-disease-fold_4-val.csv"
dev_es_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/disease/kfold_5/fold_1/val/es-disease-fold_1-val.csv"
dev_it_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/disease/kfold_5/fold_1/val/it-disease-fold_1-val.csv"
dev_ro_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/disease/kfold_5/fold_1/val/ro-disease-fold_1-val.csv"
dev_cz_disease = "/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/disease/kfold_5/fold_1/val/cz-disease-fold_1-val.csv"


## Load CSVs into DataFrames

Read the csv files to pandas dataframes, specifying the delimiter as ','.

Column layout: `ner`, `start`, `end`, `token`.


In [ ]:
pd.options.display.max_colwidth = 900000

column_names = ["ner", "start", "end", "token"]

train_en_disease_df = pd.read_csv(train_en_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
train_nl_disease_df = pd.read_csv(train_nl_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
train_sv_disease_df = pd.read_csv(train_sv_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
train_es_disease_df = pd.read_csv(train_es_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
train_it_disease_df = pd.read_csv(train_it_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
train_ro_disease_df = pd.read_csv(train_ro_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
train_cz_disease_df = pd.read_csv(train_cz_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")

dev_en_disease_df = pd.read_csv(dev_en_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
dev_nl_disease_df = pd.read_csv(dev_nl_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
dev_sv_disease_df = pd.read_csv(dev_sv_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
dev_es_disease_df = pd.read_csv(dev_es_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
dev_it_disease_df = pd.read_csv(dev_it_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
dev_ro_disease_df = pd.read_csv(dev_ro_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")
dev_cz_disease_df = pd.read_csv(dev_cz_disease, delimiter=",", names=column_names, skip_blank_lines=False, encoding="utf-8")


## Merge train + validation and keep `token` / `ner`

Training uses all train+val data. Thus, we merge the train and validation sets adding empty rows that separate corpora so sentence grouping resets at boundaries. We only keep `token` and `ner` columns (in that order).


In [ ]:
empty_row = pd.DataFrame([[None] * len(train_en_disease_df.columns)], columns=train_en_disease_df.columns)

train_multilingual_disease_df = pd.concat(
    [
        train_en_disease_df, empty_row, dev_en_disease_df, empty_row,
        train_nl_disease_df, empty_row, dev_nl_disease_df, empty_row,
        train_sv_disease_df, empty_row, dev_sv_disease_df, empty_row,
        train_es_disease_df, empty_row, dev_es_disease_df, empty_row,
        train_it_disease_df, empty_row, dev_it_disease_df, empty_row,
        train_ro_disease_df, empty_row, dev_ro_disease_df, empty_row,
        train_cz_disease_df, empty_row, dev_cz_disease_df], ignore_index=True)

train_multilingual_disease_df = train_multilingual_disease_df[["token", "ner"]].copy()
train_df = train_multilingual_disease_df


## Sentence-level table (`id`, `tokens`, `ner`)

We need to have a sentence for each id (not a single token). For this, we modify the dataframes:

In [ ]:
ids, tokens, ners = [], [], []
current_id = 0
current_tokens, current_ners = [], []

for _, row in train_df.iterrows():
    token, ner = row["token"], row["ner"]
    if pd.notna(token):
        current_tokens.append(str(token))
        current_ners.append(str(ner))
    else:
        if current_tokens:
            ids.append(current_id)
            tokens.append(current_tokens)
            ners.append(current_ners)
        current_id += 1
        current_tokens, current_ners = [], []

new_train_df = pd.DataFrame({"id": ids, "tokens": tokens, "ner": ners})


## Hugging Face `DatasetDict` and label maps

We now convert the dataframe to a HuggingFace DatasetDict.

`id2label` / `label2id` use **sorted** unique labels so IDs are stable across runs.


In [ ]:
train_dataset = Dataset.from_pandas(new_train_df)
dataset = DatasetDict({"train": train_dataset})

all_ner_labels = []
for row in dataset["train"]:
    for label in row["ner"]:
        all_ner_labels.append(label)

label_names = sorted(set(all_ner_labels))
id2label = {i: lab for i, lab in enumerate(label_names)}
label2id = {lab: i for i, lab in enumerate(label_names)}

train_ner_ids = [[label2id[lab] for lab in row["ner"]] for row in dataset["train"]]

updated_train = Dataset.from_dict(
    {"id": dataset["train"]["id"], "tokens": dataset["train"]["tokens"], "ner": train_ner_ids}
)
new_dataset = DatasetDict({"train": updated_train})


## Tokenizer and label alignment

`model_checkpoint` can be swapped if you keep `num_labels` and label maps consistent.


In [ ]:
model_checkpoint = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )
    labels = []
    for i, label in enumerate(examples["ner"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


tokenized_dataset = new_dataset.map(tokenize_and_align_labels, batched=True)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


## Model, trainer, and training


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)


Define model trainer and hyperparameters:

In [ ]:
torch.cuda.empty_cache()

wandb.init(settings=wandb.Settings(init_timeout=120), mode="offline")

training_args = TrainingArguments(
    output_dir="XLM-R_multilingual_disease",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="epoch",
    load_best_model_at_end=False,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
    processing_class=tokenizer,
)


Train the model:

In [ ]:
trainer.train()


Push the trained model to HF hub:

In [ ]:
trainer.push_to_hub()
